# 02 -- Data Preparation for Modeling

Continuing from `01_eda.ipynb`: randomization held (no covariate/group SMD flagged), `history_segment` is a clean non-overlapping binning of `history`, and the one-hot design matrix is full rank with `drop_first=True`. Taking those as given, this notebook turns the checked raw data into model-ready train/val/test splits for `03_uplift_models.ipynb`.

In [1]:
%pip install pandas numpy scikit-learn pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 5.6 MB/s  0:00:09m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(project_root))

from src.data_prep import (
    load_hillstrom,
    check_design_matrix_rank,
    build_model_ready_frame,
    stratified_train_val_test_split,
    make_binary_treatment,
    save_processed_splits,
)

pd.set_option('display.max_columns', None)

df = load_hillstrom('../data/raw/hillstrom.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.440002,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.079987,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.649994,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.830017,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.340000,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


## Step 1: Train / validation / test split

Stratifying only on `segment`: this is a randomized experiment, so each split needs to stay a valid comparison across the three arms on its own, not accidentally treatment- or control-heavy. `stratified_train_val_test_split` chains two stratified `train_test_split` calls, both on `segment`, so the per-arm ratio holds in train, val, and test independently.

Not stratifying on `visit`/`conversion` too: conversion is rare (~0.9%, ~576 events over 64k rows), so compounding the stratification key with outcome would mean 6+ strata for a rate that already lands close to the population value by sample size alone (checked below rather than assumed). It would also bake in an assumption about which arm-pair / outcome combination matters most, ahead of `make_binary_treatment` (next notebook) actually reducing `segment` to a specific binary comparison.

60/20/20 split: Qini/AUUC are noisier than standard classification metrics since they depend on the rare outcome and on ranking a continuous score, so a small test set is mostly noise. This gives ~12,800 rows each for val (model/hyperparameter selection across the S-/T-/X-Learner candidates from `decisions_log.md`) and test (final evaluation), 38,400 for train.

## Step 2: Encoding covariates

`zip_code` and `channel`: one-hot, `drop_first=True`. Both are 3-level nominal categoricals; one-hot works across linear/logistic base learners and sklearn's tree ensembles (`RandomForestClassifier`, `GradientBoostingClassifier`), none of which take a native `category` dtype. `drop_first=True` keeps the design matrix full rank, per the `check_design_matrix_rank` reuse below. No parallel integer/native-categorical encoding for LightGBM/CatBoost/`HistGradientBoostingClassifier` -- if the next notebook picks one of those, deriving an integer code from the one-hot columns is a one-line change.

`history_segment`: dropped. `01_eda.ipynb`'s `check_history_segment_consistency` showed it's a non-overlapping binning of `history` -- keeping both would dilute SHAP importance across two correlated columns and reintroduce the VIF redundancy already flagged in EDA for linear learners. `history` stays as the finer-grained continuous version.

Encoding runs on the full dataset before splitting: deterministic per-row mapping from a category dtype already fixed by `load_hillstrom`, unlike scaling/imputation which fit parameters from data. Also guarantees identical columns across splits.

`segment` stays untouched, still 3-arm. scikit-uplift's meta-learners want a binary treatment, but this experiment has two active arms (Mens/Womens E-Mail) plus control; `make_binary_treatment` does that reduction on demand, once per analysis, rather than forcing the choice here.

In [3]:
rank_check = check_design_matrix_rank(
    df,
    numeric_covariates=['recency', 'history'],
    categorical_covariates=['zip_code', 'channel'],
    drop_first=True,
    include_intercept=True,
)
assert rank_check['full_rank'], f'Design matrix is not full rank: {rank_check}'
print('Full-rank check (reused from EDA):', rank_check)

model_ready_df = build_model_ready_frame(df)

assert model_ready_df.isna().sum().sum() == 0, 'Unexpected missing values after encoding.'
print('Shape:', model_ready_df.shape)
print('Columns:', list(model_ready_df.columns))
model_ready_df.dtypes

Full-rank check (reused from EDA): {'n_columns': 7, 'rank': 7, 'full_rank': True}
Shape: (64000, 13)
Columns: ['recency', 'history', 'mens', 'womens', 'newbie', 'segment', 'visit', 'conversion', 'spend', 'zip_code_Surburban', 'zip_code_Urban', 'channel_Phone', 'channel_Web']


recency                   int8
history                float32
mens                      int8
womens                    int8
newbie                    int8
segment               category
visit                     int8
conversion                int8
spend                  float32
zip_code_Surburban        int8
zip_code_Urban            int8
channel_Phone             int8
channel_Web               int8
dtype: object

## Step 3: How much feature engineering belongs here? Very little.

No interaction terms, no polynomial features, no manual binning beyond the raw data. Finding interactions is what the uplift models in `03_uplift_models.ipynb` are for -- hand-engineering them here would mean guessing at treatment-effect modulators before the modeling/SHAP stage gets to find them.

No scaling: scaler parameters must be fit on training folds only, not once over the full dataset or even once over `train_df` reused across CV folds. That belongs inside a `Pipeline`/`ColumnTransformer` in the modeling notebook, fit fresh per fold.

No imputation: `01_eda.ipynb`'s missingness check found zero missing values.

In [4]:
train_df, val_df, test_df = stratified_train_val_test_split(
    model_ready_df, treatment_col='segment', test_size=0.2, val_size=0.2, random_state=42,
)

split_sizes = pd.DataFrame({
    'n_rows': {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)},
})
print(split_sizes)

segment_props = pd.DataFrame({
    'full_data': df['segment'].value_counts(normalize=True),
    'train': train_df['segment'].value_counts(normalize=True),
    'val': val_df['segment'].value_counts(normalize=True),
    'test': test_df['segment'].value_counts(normalize=True),
}).round(4)
print('Segment proportions by split:')
display(segment_props)

outcome_rates = pd.DataFrame({
    'full_data': df[['visit', 'conversion']].mean(),
    'train': train_df[['visit', 'conversion']].mean(),
    'val': val_df[['visit', 'conversion']].mean(),
    'test': test_df[['visit', 'conversion']].mean(),
}).round(4)
print('Outcome rates by split (not stratified on, checked for drift anyway):')
display(outcome_rates)

       n_rows
train   38400
val     12800
test    12800
Segment proportions by split:


,full_data,train,val,test
segment,,,,
Womens E-Mail,0.3342,0.3342,0.3342,0.3341
Mens E-Mail,0.3329,0.3329,0.3329,0.3330
No E-Mail,0.3329,0.3329,0.3329,0.3329


Outcome rates by split (not stratified on, checked for drift anyway):


,full_data,train,val,test
visit,0.1468,0.1480,0.1473,0.1427
conversion,0.0090,0.0084,0.0099,0.0099


### Treatment column demo

A preview of `make_binary_treatment`, showing the reduction the next notebook will apply once per arm. Not saved from this notebook -- `segment` stays 3-arm in the files written below.

In [5]:
mens_vs_control = make_binary_treatment(train_df, treatment_group='Mens E-Mail')
womens_vs_control = make_binary_treatment(train_df, treatment_group='Womens E-Mail')

print('Mens E-Mail vs No E-Mail (train):', mens_vs_control.shape[0], 'rows')
print(mens_vs_control['treatment'].value_counts())
print('Womens E-Mail vs No E-Mail (train):', womens_vs_control.shape[0], 'rows')
print(womens_vs_control['treatment'].value_counts())

Mens E-Mail vs No E-Mail (train): 25568 rows
treatment
0    12784
1    12784
Name: count, dtype: int64
Womens E-Mail vs No E-Mail (train): 25616 rows
treatment
1    12832
0    12784
Name: count, dtype: int64


## Step 4: Persisting the processed splits

Three separate files (`train.parquet`, `val.parquet`, `test.parquet`) rather than one with a `split` column -- the file boundary itself prevents accidentally loading val/test rows into training, no filter condition to get wrong later.

Parquet over CSV: `model_ready_df` has `int8` flags/dummies and a `category` `segment` column that CSV would flatten to strings/generic types on reload, needing re-casting logic that `load_hillstrom` already exists to centralize for the raw data.

`save_processed_splits` also writes `feature_manifest.json` (feature/outcome/treatment column names, per-split segment proportions), so the column list lives in one place instead of being retyped and drifting out of sync later.

In [6]:
feature_columns = [
    c for c in model_ready_df.columns if c not in ['segment', 'visit', 'conversion', 'spend']
]

save_processed_splits(
    train_df, val_df, test_df,
    output_dir='../data/processed',
    feature_columns=feature_columns,
    outcome_columns=['visit', 'conversion', 'spend'],
    treatment_col='segment',
)

import json

output_dir = Path('../data/processed')
print('Files written:', sorted(p.name for p in output_dir.iterdir()))

with open(output_dir / 'feature_manifest.json') as f:
    manifest = json.load(f)
manifest

Files written: ['.gitkeep', 'feature_manifest.json', 'test.parquet', 'train.parquet', 'val.parquet']


{'feature_columns': ['recency',
  'history',
  'mens',
  'womens',
  'newbie',
  'zip_code_Surburban',
  'zip_code_Urban',
  'channel_Phone',
  'channel_Web'],
 'outcome_columns': ['visit', 'conversion', 'spend'],
 'treatment_col': 'segment',
 'split_sizes': {'train': 38400, 'val': 12800, 'test': 12800},
 'segment_proportions': {'train': {'Womens E-Mail': 0.3342,
   'Mens E-Mail': 0.3329,
   'No E-Mail': 0.3329},
  'val': {'Womens E-Mail': 0.3342, 'Mens E-Mail': 0.3329, 'No E-Mail': 0.3329},
  'test': {'Womens E-Mail': 0.3341,
   'Mens E-Mail': 0.333,
   'No E-Mail': 0.3329}}}

## Summary

`train.parquet` / `val.parquet` / `test.parquet` under `data/processed/`: one-hot `zip_code`/`channel`, `history_segment` dropped, `segment` left 3-arm, outcomes untouched, stratified 60/20/20 on `segment`. `feature_manifest.json` carries the column names and per-split segment proportions.

Next: `03_uplift_models.ipynb` loads `train.parquet`/`val.parquet`, calls `make_binary_treatment` per arm, fits the S-/T-/X-Learner candidates from `decisions_log.md`. `test.parquet` stays untouched until final evaluation.